# Craft Features from Linter Analysis
I need to create a dataset with the following features from the linter analysis to use them later to train a random forest model.
| Feature                  | Description                                                                 | Type    |
| :----------------------: | :-------------------------------------------------------------------------: | :-----: |
| Issue Code               |  Error/Warning Code e.g. PERF401                                            | String  |
| Row                      |  Line Position                                                              | Int     |
| Column                   |  Line Position                                                              | Int     |
| Fix Availability         |  Whether automatic fixes are available                                      | Boolean |
| Fix Applicability        |  Safe/Unsafe                                                                | String  |
| Fix Complexity           |  Number of needed edits (length of edits array)                             | Int     |
| Fix Description          |  Message Text e.g. “"Use a list comprehension to create a transformed list” | String  |
| Issue Frequency          |  Count of each issue type                                                   | Int     |
| Issues Per Line          |  Issues per line of code in a file                                          | Int     |
<!-- | Characters Per Line      |  Number of chars per line                                                   | Int     | -->


In [2]:
import glob
import json
import pandas as pd

In [6]:
file_paths = glob.glob('lint_results/python/*.py.json')
file_paths[0]

'lint_results/python/SportsRank.py.json'

In [28]:
columns = ["title",
           "issue_code", 
           "line_row", 
           "line_column", 
           "fix_availability", 
           "fix_applicability", 
           "fix_complexity",
           "fix_description",
           "issue_frequency",
           "issues_per_line"]
linter_analysis_features = pd.DataFrame([], columns=columns)

In [31]:
for file_path in file_paths[:1]:
    with open(file_path, 'r') as file:
        linter_analysis_objects = json.load(file)
    title = file_path.replace('.py.json','').replace('lint_results/python/','')
    for linter_obj in linter_analysis_objects:
        issue_code = linter_obj['code']
        row = linter_obj['end_location']['row']
        column = linter_obj['end_location']['column'] 
        fix = linter_obj['fix']
        fix_availability = fix is not None
        fix_applicability = None
        fix_complexity = None
        if fix_availability:
            fix_applicability = fix['applicability']
            fix_complexity = len(fix['edits'])
            
        fix_description = linter_obj['message']
        issue_frequency = sum(obj['code'] == issue_code for obj in linter_analysis_objects)
        issues_per_line = sum(obj['end_location']['row'] == row for obj in linter_analysis_objects)
        linter_analysis_features.loc[len(linter_analysis_features)] = [
            title,
            issue_code,
            row,
            column,
            fix_availability,
            fix_applicability,
            fix_complexity,
            fix_description,
            issue_frequency,
            issues_per_line
        ]
linter_analysis_features.head()

,title,issue_code,line_row,line_column,fix_availability,fix_applicability,fix_complexity,fix_description,issue_frequency,issues_per_line
0,SportsRank,PLC0415,2,15,False,None,None,`import` should be at the top-level of a file,1,1
1,SportsRank,A001,3,10,False,None,None,Variable `input` is shadowing a Python builtin,1,1
2,SportsRank,E302,16,4,True,safe,1,"Expected 2 blank lines, found 0",4,2
3,SportsRank,ANN001,16,31,False,None,None,Missing type annotation for function argument ...,4,2
4,SportsRank,E302,25,4,True,safe,1,"Expected 2 blank lines, found 0",4,2


In [32]:
linter_analysis_features.to_csv('datasets/python_linter_analysis_features.csv', index=False)

### Features Per Line not Per Issue

In [13]:
crafted_features = pd.read_csv('datasets/python_linter_analysis_features.csv')

In [15]:
cleaned_crafted_features = crafted_features.drop(columns=['line_column', 'issue_frequency', 'fix_description'])

In [18]:
cleaned_crafted_features.dtypes

title                 object
issue_code            object
line_row               int64
fix_availability        bool
fix_applicability     object
fix_complexity       float64
issues_per_line        int64
dtype: object

In [70]:
grouped_crafted_features = cleaned_crafted_features.groupby(['title', 'line_row']).agg(
    any_fix_available=('fix_availability', 'any'),
    all_fixes_available=('fix_availability', 'all'),
    fix_availability_ratio=('fix_availability', 'mean'),

    unsafe_fixes=('fix_applicability', lambda x: (x == 'unsafe').sum()),
    safe_fixes=('fix_applicability', lambda x: (x == 'safe').sum()),

    avg_fix_complexity=('fix_complexity', 'mean'),
    max_fix_complexity=('fix_complexity', 'max'),
    min_fix_complexity=('fix_complexity', 'min'),
    sum_fix_complexity=('fix_complexity', 'sum'),

    unique_issue_types=('issue_code', pd.Series.nunique),
    issue_types_list=('issue_code', lambda x: list(set(x))),

    issues_per_line=('issues_per_line', 'first') 
).round(2).reset_index()

In [68]:
grouped_crafted_features.columns = [col for col in grouped_crafted_features.columns]
final_crafted_features = grouped_crafted_features.reset_index().drop('index', axis=1)

In [69]:
final_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,sum_fix_complexity,unique_issue_types,issue_types_list,issues_per_line
0,SportsRank,2,False,False,0.00,0,0,NaN,NaN,NaN,0.0,1,[PLC0415],1
1,SportsRank,3,False,False,0.00,0,0,NaN,NaN,NaN,0.0,1,[A001],1
2,SportsRank,16,True,False,0.50,0,3,1.0,1.0,1.0,3.0,2,"[ANN001, E302]",2
3,SportsRank,25,True,False,0.50,0,3,1.0,1.0,1.0,3.0,2,"[ANN001, E302]",2
4,SportsRank,34,True,False,0.33,0,3,1.0,1.0,1.0,3.0,2,"[ANN001, E302]",3
